# 04 Trasforma e combina

## Setup

In [142]:
import numpy as np
import pandas as pd

In [143]:
titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

In [144]:
titanic

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## Creare colonne separate

In [145]:
titanic["family_size"] = (titanic["SibSp"]) + (titanic["Parch"]) + 1 # Questa è la più semplice, non necessita di metodi come map/replace, np.where o apply

In [146]:
titanic["is_alone"] = titanic["family_size"] == 1 # Ho usato np.where perchè la consegna richiede l'uso di un metodo condizionale, e qui ci sono 2 condizioni. Se fossero state più condizioni avrei utilizzato .apply

In [147]:
titanic["embarked_full"] = titanic["Embarked"].map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}) # Qui andava bene sia map che replace visto che in ogni caso il valore mancante rimane NaN. Per sicurezza ho usato map

In [148]:
def fare_calc(fare):
    if fare < 10:
        return "low"
    elif fare <= 50:
        return "medium"
    else:
        return "high"

titanic["fare_level"] = titanic["Fare"].apply(fare_calc)

# Essendoci 3 condizioni ho utilizzato .apply(), creando una funzione apposita per la condizionale

In [149]:
titanic["family_size"].value_counts()

family_size
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64

In [150]:
titanic["is_alone"].value_counts()

is_alone
True     537
False    354
Name: count, dtype: int64

In [151]:
titanic["embarked_full"].value_counts()

embarked_full
Southampton    644
Cherbourg      168
Queenstown      77
Name: count, dtype: int64

In [152]:
titanic["fare_level"].value_counts()

fare_level
medium    395
low       336
high      160
Name: count, dtype: int64

## Trasformare valori esistenti

In [153]:
titanic = titanic.rename(columns={"SibSp": "siblings_spouses", "Parch": "parents_children"})

In [154]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,siblings_spouses,parents_children,Ticket,Fare,Cabin,Embarked,family_size,is_alone,embarked_full,fare_level
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,Southampton,low
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,False,Cherbourg,high
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,True,Southampton,low
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,False,Southampton,high
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,True,Southampton,low


In [155]:
titanic.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age',
       'siblings_spouses', 'parents_children', 'Ticket', 'Fare', 'Cabin',
       'Embarked', 'family_size', 'is_alone', 'embarked_full', 'fare_level'],
      dtype='str')

In [156]:
titanic["Pclass"] = titanic["Pclass"].astype("category")

In [157]:
titanic["Pclass"].dtype

CategoricalDtype(categories=[1, 2, 3], ordered=False, categories_dtype=int64)

In [158]:
titanic["fare_level_v2"] = pd.cut(
    titanic["Fare"],
    bins=[-0.01, 10, 50, float("inf")],
    labels=["low", "medium", "high"]
)

In [159]:
titanic["fare_level_v2"].value_counts()

fare_level_v2
medium    395
low       336
high      160
Name: count, dtype: int64

## Combinare DataFrame

In [160]:
'''
1) concat → "incolla" DataFrame uno sotto l'altro (o uno accanto all'altro).
Serve quando hanno la stessa struttura.

2) merge → unisce due DataFrame sulla base di una colonna comune (la "chiave").
È il SQL JOIN di pandas.

3) join → variante di merge che lavora sull'indice invece che sulle colonne. 
La userai raramente, la menziono solo per completezza.
'''

'\n1) concat → "incolla" DataFrame uno sotto l\'altro (o uno accanto all\'altro).\nServe quando hanno la stessa struttura.\n\n2) merge → unisce due DataFrame sulla base di una colonna comune (la "chiave").\nÈ il SQL JOIN di pandas.\n\n3) join → variante di merge che lavora sull\'indice invece che sulle colonne. \nLa userai raramente, la menziono solo per completezza.\n'

In [161]:
''' varianti dell'utilizzo della funzione merge()

how -> cosa tiene

1) inner -> Solo righe con chiave presente in entrambe le tabelle

2) left -> Tutte le righe della tabella sinistra + match dalla destra 
(NaN dove non c'è match)

3) right -> Tutte le righe della destra + match della sinistra

4) outer -> Tutte le righe di entrambe (NaN dove non c'è match)
'''

" varianti dell'utilizzo della funzione merge()\n\nhow -> cosa tiene\n\n1) inner -> Solo righe con chiave presente in entrambe le tabelle\n\n2) left -> Tutte le righe della tabella sinistra + match dalla destra \n(NaN dove non c'è match)\n\n3) right -> Tutte le righe della destra + match della sinistra\n\n4) outer -> Tutte le righe di entrambe (NaN dove non c'è match)\n"

In [162]:
ports = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "port_country": ["United Kingdom", "France", "Ireland"],
    "port_latitude": [50.90, 49.63, 51.85]
})

In [163]:
ports

,Embarked,port_country,port_latitude
0,S,United Kingdom,50.90
1,C,France,49.63
2,Q,Ireland,51.85


In [164]:
merged = titanic.merge(ports, on="Embarked", how="left")

In [165]:
merged.shape # Numero di righe rimasto uguale

(891, 19)

In [166]:
merged["port_country"].isna().sum() # 2 passeggeri hanno valori mancanti sul port country

np.int64(2)

In [167]:
merged["port_country"].value_counts()

port_country
United Kingdom    644
France            168
Ireland            77
Name: count, dtype: int64

In [168]:
titanic_first_half = titanic.iloc[:500]
titanic_second_half = titanic.iloc[500:]

In [169]:
t_combined = pd.concat([titanic_first_half, titanic_second_half], axis=0)

In [170]:
t_combined.shape

(891, 17)

In [171]:
titanic.shape

(891, 17)

In [172]:
t_combined.reset_index(drop=True)

,PassengerId,Survived,Pclass,Name,Sex,Age,siblings_spouses,parents_children,Ticket,Fare,Cabin,Embarked,family_size,is_alone,embarked_full,fare_level,fare_level_v2
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,Southampton,low,low
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,False,Cherbourg,high,high
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,True,Southampton,low,low
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,False,Southampton,high,high
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,True,Southampton,low,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,1,True,Southampton,medium,medium
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,1,True,Southampton,medium,medium
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,4,False,Southampton,medium,medium
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,1,True,Cherbourg,medium,medium


## Riepilogo

In [173]:
# Riepilogo — Trasformare e combinare

## 1. Creare colonne derivate
'''
In pandas creo una nuova colonna assegnandola come se fosse una chiave di un dizionario: `df["nuova"] = ...`. La domanda vera è *cosa* metto a destra dell'uguale. Gerarchia degli strumenti, dal più veloce/semplice al più lento/flessibile:

1. **Operazione vettoriale diretta** — prima scelta quando la trasformazione è matematica o logica sull'intera colonna. Es. `df["family_size"] = df["SibSp"] + df["Parch"] + 1`. Se la condizione produce già True/False, non serve avvolgerla in `np.where`: `df["is_alone"] = df["family_size"] == 1` basta.
2. **`map` su una Series** — rimappare valori discreti con un dizionario. Se un valore non è nel dizionario → `NaN`.
3. **`replace` su una Series** — come `map` ma i valori non trovati restano invariati.
4. **`np.where(cond, A, B)`** — scelta binaria vettoriale, equivalente di un if/else.
5. **`apply(funzione)`** — ultima spiaggia, quando la logica è troppo complessa per gli altri. È la più lenta perché sotto fa un loop Python.
6. **`pd.cut` / `pd.qcut`** — per il binning di colonne numeriche. `cut` con confini che decido io, `qcut` con lo stesso numero di osservazioni per bin. Più pulito di `apply` con if/elif/else quando si tratta di assegnare etichette per soglie.

## 2. Trasformare valori esistenti

- **`rename(columns={...})`** — rinominare colonne. Restituisce un nuovo DataFrame, va riassegnato.
- **`astype("category")`** — cambiare il dtype. Utile per colonne categoriche (es. `Pclass`): risparmia memoria e rende esplicita la natura della variabile. Se voglio un ordine (1ª < 2ª < 3ª) uso `pd.CategoricalDtype(categories=[...], ordered=True)`.

## 3. Combinare DataFrame

- **`concat([df1, df2], axis=0)`** — incolla DataFrame uno sotto l'altro (o di fianco con `axis=1`). Uso tipico: file mensili da unire in un dataset annuale. Dopo un concat fare sempre `reset_index(drop=True)`, altrimenti si rischiano indici duplicati.
- **`merge(..., on="chiave", how=...)`** — è il JOIN di SQL in pandas. I 4 tipi di `how`:
  - `inner` → solo righe con chiave presente in **entrambe**
  - `left` → tutte le righe della sinistra, NaN dove la destra non ha match
  - `right` → speculare di `left`
  - `outer` → unione di entrambe
- **Duplicati inaspettati dopo merge**: se la chiave non è univoca in una delle due tabelle, il merge moltiplica le righe. Controllo preventivo: `chiave.duplicated().sum()`. Controllo difensivo: confrontare `.shape` prima e dopo il merge.

## 4. Gotchas incontrate

- **`map` vs `replace` sui valori non previsti**: `map` → NaN, `replace` → valore originale invariato. Nel caso di `Embarked` → `embarked_full` il risultato coincideva perché i valori "mancanti" erano già NaN in origine, ma in generale la differenza conta.
- **2 NaN che si propagano dal merge**: `Embarked` aveva 2 NaN nel dataset originale. Con `how="left"` il merge li tiene come righe ma `port_country` resta NaN perché `ports` non ha una riga con chiave NaN. Somma `value_counts` + `isna().sum()` = 891.
- **`[]` vs `()` in una condizione**: `[cond]` crea una lista di un elemento, non raggruppa. Per raggruppare una condizione (es. dentro `np.where` o in una maschera booleana) servono le tonde: `(df["a"] + df["b"]) == 0`.
- **`how="left"` tiene la sinistra, non entrambe**: chi non ha match nella sinistra non compare nel risultato, anche se ha righe nella destra. Esempio: Carla senza ordini sparisce con `how="left"` applicato a `ordini.merge(clienti, ...)`.
- **`np.where` con uscite `True`/`False`**: se la condizione produce già booleani, `np.where(cond, True, False)` è ridondante. La condizione da sola basta.
- **Soglie inclusive/esclusive in `apply` per binning**: `elif fare < 50` esclude 50, `elif fare <= 50` lo include. Rileggere sempre la consegna prima di scrivere le soglie.
- **`value_counts()` su singola colonna vs su più colonne**: `df[["a","b"]].value_counts()` dà le combinazioni, non le distribuzioni singole. Per quelle, chiamare `value_counts()` una colonna alla volta.

## 5. Metodo di lavoro (nota per me stesso)

Oggi è emerso di nuovo il punto debole già segnalato: leggere la consegna di fretta e saltare pezzi. Riflesso da costruire: **prima di rispondere, contare le richieste del messaggio e verificare di aver coperto tutte**. Stessa cosa sul merge: **prima di lanciarlo, stimare mentalmente quante righe mi aspetto; poi controllare con `.shape`**. Se il numero non torna, c'è un bug nei dati o nella mia comprensione.
'''

'\nIn pandas creo una nuova colonna assegnandola come se fosse una chiave di un dizionario: `df["nuova"] = ...`. La domanda vera è *cosa* metto a destra dell\'uguale. Gerarchia degli strumenti, dal più veloce/semplice al più lento/flessibile:\n\n1. **Operazione vettoriale diretta** — prima scelta quando la trasformazione è matematica o logica sull\'intera colonna. Es. `df["family_size"] = df["SibSp"] + df["Parch"] + 1`. Se la condizione produce già True/False, non serve avvolgerla in `np.where`: `df["is_alone"] = df["family_size"] == 1` basta.\n2. **`map` su una Series** — rimappare valori discreti con un dizionario. Se un valore non è nel dizionario → `NaN`.\n3. **`replace` su una Series** — come `map` ma i valori non trovati restano invariati.\n4. **`np.where(cond, A, B)`** — scelta binaria vettoriale, equivalente di un if/else.\n5. **`apply(funzione)`** — ultima spiaggia, quando la logica è troppo complessa per gli altri. È la più lenta perché sotto fa un loop Python.\n6. **`pd.cut`